# SAFE ML Txns Endpoint — Deploy Notebook

Deploys the `SAFE_TXNS_ENDPOINT_DEV` SageMaker endpoint with:
- K-Means clustering (8 clusters)
- Statistical rules v8 (R1–R12)
- Similarity matching via Athena (cross-account us-east-2)

**Run order:** Step 1 (once, if tarball not in S3) → Step 2 (once, if tarball not in S3) → Step 3 (deploy) → Step 4 (test)

In [ ]:
import boto3
import os

BUCKET        = 'blossom-analytics-safe-dev-nv'
ENDPOINT_NAME = 'SAFE_TXNS_ENDPOINT_DEV'
REGION        = 'us-east-1'
MODEL_S3_KEY  = 'safe_txns/similarity/endpoint/v2/model.tar.gz'
MODEL_S3_URI  = f's3://{BUCKET}/{MODEL_S3_KEY}'

s3 = boto3.client('s3')
print('Config OK — endpoint:', ENDPOINT_NAME)
print('Model URI:', MODEL_S3_URI)

### (Skip if tarball already in S3) Step 1: Build model.tar.gz

Creates the tarball with **code + model artifacts** required by the SageMaker container.

Contents:
- `inference_rules.py`, `similarity_matcher.py`, `schema_validator.py`, `statistical_rules.py`, `requirements.txt` (endpoint code)
- `kmeans_model.joblib`, `preprocessing_pipeline.joblib`, `centroids.csv`, `selected_features.csv`, `kmeans_artifacts.json` (model artifacts)

In [ ]:
import tarfile, os

# Current S3 paths for model artifacts (V2)
MODEL_ARTIFACTS = {
    'kmeans_model.joblib':          'safe_txns/kmeans/kmeans_analysis/V2/artifact/kmeans_model.joblib',
    'kmeans_artifacts.json':        'safe_txns/kmeans/kmeans_analysis/V2/artifact/kmeans_artifacts.json',
    'centroids.csv':                'safe_txns/kmeans/centroids/V2/centroids.csv',
    'preprocessing_pipeline.joblib':'safe_txns/preprocessing/NOVEMBER/preprocessing_pipeline.joblib',
    'selected_features.csv':        'safe_txns/feature_selection/NOVEMBER/selected_features.csv',
}

# Code files (relative to repo root — run !pwd to confirm)
CODE_FILES = [
    'endpoint/inference_rules.py',
    'endpoint/similarity_matcher.py',
    'endpoint/schema_validator.py',
    'endpoint/statistical_rules.py',
    'endpoint/requirements.txt',
]

LOCAL_ARTIFACTS = '/tmp/sm_artifacts'
TAR_PATH        = '/tmp/model.tar.gz'

os.makedirs(LOCAL_ARTIFACTS, exist_ok=True)

# Download model artifacts
print('Downloading model artifacts...')
for filename, s3_key in MODEL_ARTIFACTS.items():
    local_path = os.path.join(LOCAL_ARTIFACTS, filename)
    s3.download_file(BUCKET, s3_key, local_path)
    size_kb = os.path.getsize(local_path) // 1024
    print(f'  ✓ {filename} ({size_kb} KB)')

# Build tarball: code + artifacts flat at root
print('\nBuilding tarball...')
with tarfile.open(TAR_PATH, 'w:gz') as tar:
    for code_file in CODE_FILES:
        if os.path.exists(code_file):
            tar.add(code_file, arcname=os.path.basename(code_file))
            print(f'  + {code_file}')
        else:
            print(f'  ✗ MISSING: {code_file}  ← run from repo root!')
    for filename in MODEL_ARTIFACTS:
        tar.add(os.path.join(LOCAL_ARTIFACTS, filename), arcname=filename)
        print(f'  + {filename}')

size_kb = os.path.getsize(TAR_PATH) // 1024
print(f'\n✅ model.tar.gz created ({size_kb} KB) at {TAR_PATH}')

### (Skip if tarball already in S3) Step 2: Upload model.tar.gz to S3

In [ ]:
s3.upload_file(TAR_PATH, BUCKET, MODEL_S3_KEY)
print(f'✅ Uploaded to {MODEL_S3_URI}')

### Step 3: Deploy endpoint

- **Endpoint name:** `SAFE_TXNS_ENDPOINT_DEV`
- **Instance:** ml.m5.large
- **Framework:** scikit-learn 1.2-1
- **Entry point:** `inference_rules.py`
- **Athena:** `dlh_silver_safe_alpha.safetransactionresults` (us-east-2)
- **Data:** Nov 2023 – Nov 2025 (168k txns), K-Means 8 clusters

In [ ]:
!pwd  # Must be repo root (safe_txns_sim_endpoint/)

In [ ]:
from sagemaker.sklearn.model import SKLearnModel
from sagemaker import get_execution_role, Session
import sagemaker

sagemaker_session = sagemaker.Session()
role = get_execution_role()
print('Role:', role)

sk_model = SKLearnModel(
    model_data=MODEL_S3_URI,
    role=role,
    entry_point='inference_rules.py',
    framework_version='1.2-1',
    py_version='py3',
    sagemaker_session=sagemaker_session,
    env={
        # Similarity matching via Athena (cross-account us-east-2)
        'SIMILARITY_THRESHOLD':           '0.90',
        'SIMILARITY_ATHENA_DATABASE':     'dlh_silver_safe_alpha',
        'SIMILARITY_ATHENA_TABLE':        'safetransactionresults',
        'SIMILARITY_ATHENA_S3_STAGING':   's3://blossom-analytics-datalake-alpha/datalake/gold/athena-metadata/',
        'SIMILARITY_ATHENA_REGION':       'us-east-2',
        'ATHENA_WINDOW_MONTHS':           '6',
        'ATHENA_TIMEOUT_SECONDS':         '10',
    }
)
print('Model configured ✅')

In [ ]:
predictor = sk_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=ENDPOINT_NAME,
    wait=True,
)
print(f'\n✅ Endpoint {ENDPOINT_NAME} InService')

### Step 4: Test the endpoint

In [ ]:
import boto3, pandas as pd, io, json

In [ ]:
# Input file in S3 — change filename as needed
data       = 'test_escenarios.csv'
input_key  = f'safe_txns/similarity/real_time/input/{data}'
output_key = f'safe_txns/similarity/real_time/output/{data}'

In [ ]:
s3      = boto3.client('s3')
runtime = boto3.client('sagemaker-runtime', region_name=REGION)

In [ ]:
%%time
# Load input CSV from S3
response = s3.get_object(Bucket=BUCKET, Key=input_key)
df = pd.read_csv(response['Body'])
print(f'Input: {len(df)} rows')

# Invoke endpoint
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, header=True, index=False)

response = runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType='text/csv',
    Body=csv_buffer.getvalue()
)

parsed = json.loads(response['Body'].read().decode('utf-8'))
print(f'Output: {len(parsed)} results')

In [ ]:
df_result = pd.DataFrame(parsed)
df_result[['TransactionID', 'idOLBUserTxns', 'decision', 'score',
           'sim_match_txn_id', 'sim_score', 'sim_status', 'sim_decision']]

In [ ]:
# Save results to S3
output_buffer = io.StringIO()
df_result.to_csv(output_buffer, index=False, na_rep='None')
s3.put_object(Bucket=BUCKET, Key=output_key, Body=output_buffer.getvalue())
print(f'✅ Results saved to s3://{BUCKET}/{output_key}')

### Check endpoint status

In [ ]:
import time
sm = boto3.client('sagemaker', region_name=REGION)

for _ in range(40):
    d  = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    st = d['EndpointStatus']
    print('status:', st)
    if d.get('FailureReason'):
        print('FailureReason:', d['FailureReason'])
        break
    if st in ('InService', 'Failed'):
        break
    time.sleep(15)

### Delete endpoint

In [ ]:
sm = boto3.client('sagemaker', region_name=REGION)
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
print(f'✅ Endpoint {ENDPOINT_NAME} deleted')